In [15]:
# CELL 1: Install additional packages if needed
# Kaggle has most pre-installed, but ensure versions
!pip install -q lightgbm shap umap-learn

In [16]:
# CELL 2: Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, HDBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import warnings
warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

In [17]:
# CELL 3: Load Olist from Kaggle input
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/samihaachowdhury/olist-ecommerce/olist_customers_dataset.csv
/kaggle/input/datasets/samihaachowdhury/olist-ecommerce/olist_sellers_dataset.csv
/kaggle/input/datasets/samihaachowdhury/olist-ecommerce/olist_order_reviews_dataset.csv
/kaggle/input/datasets/samihaachowdhury/olist-ecommerce/olist_order_items_dataset.csv
/kaggle/input/datasets/samihaachowdhury/olist-ecommerce/olist_products_dataset.csv
/kaggle/input/datasets/samihaachowdhury/olist-ecommerce/olist_geolocation_dataset.csv
/kaggle/input/datasets/samihaachowdhury/olist-ecommerce/product_category_name_translation.csv
/kaggle/input/datasets/samihaachowdhury/olist-ecommerce/olist_orders_dataset.csv
/kaggle/input/datasets/samihaachowdhury/olist-ecommerce/olist_order_payments_dataset.csv
/kaggle/input/datasets/mashlyn/online-retail-ii-uci/online_retail_II.csv


In [18]:
# CELL 4: Load CSVs (corrected paths)
BASE_PATH = '/kaggle/input/datasets/samihaachowdhury/olist-ecommerce'

orders = pd.read_csv(f'{BASE_PATH}/olist_orders_dataset.csv')
customers = pd.read_csv(f'{BASE_PATH}/olist_customers_dataset.csv')
order_items = pd.read_csv(f'{BASE_PATH}/olist_order_items_dataset.csv')
order_payments = pd.read_csv(f'{BASE_PATH}/olist_order_payments_dataset.csv')
order_reviews = pd.read_csv(f'{BASE_PATH}/olist_order_reviews_dataset.csv')

In [19]:
# CELL 5: Merge and prep timestamps
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])
df = orders.merge(customers, on='customer_id', how='left')
df = df.merge(order_items[['order_id', 'price', 'freight_value']], on='order_id', how='left')
df = df.merge(order_payments.groupby('order_id')['payment_installments'].max().reset_index(), 
              on='order_id', how='left')
df = df.merge(order_reviews.groupby('order_id')['review_score'].mean().reset_index(), 
              on='order_id', how='left')
print(f"Total orders: {len(df)}")
print(f"Unique customers: {df['customer_unique_id'].nunique()}")
print(f"Date range: {df['order_purchase_timestamp'].min()} to {df['order_purchase_timestamp'].max()}")

Total orders: 113425
Unique customers: 96096
Date range: 2016-09-04 21:15:19 to 2018-10-17 17:30:18


In [20]:
# CELL 6: VIABILITY CHECK - orders per customer
orders_per_customer = df.groupby('customer_unique_id')['order_purchase_timestamp'].count()
print("Orders per customer distribution:")
print(orders_per_customer.value_counts().sort_index().head(15))
print(f"\n1 order: {(orders_per_customer == 1).sum()} ({(orders_per_customer == 1).mean()*100:.1f}%)")
print(f"2+ orders: {(orders_per_customer >= 2).sum()} ({(orders_per_customer >= 2).mean()*100:.1f}%)")
print(f"3+ orders: {(orders_per_customer >= 3).sum()}")

Orders per customer distribution:
order_purchase_timestamp
1     84151
2      9055
3      1687
4       632
5       255
6       199
7        46
8        16
9        11
10       11
11       11
12        9
13        2
14        3
15        2
Name: count, dtype: int64

1 order: 84151 (87.6%)
2+ orders: 11945 (12.4%)
3+ orders: 2890


In [21]:
# CELL 7: VIABILITY CHECK - distinct active months per customer
df['purchase_month'] = df['order_purchase_timestamp'].dt.to_period('M')
active_months = df.groupby('customer_unique_id')['purchase_month'].nunique()
print("\nDistinct active months per customer:")
print(active_months.value_counts().sort_index().head(10))

customers_3plus_months = (active_months >= 3).sum()
print(f"\nCustomers with >=3 active months: {customers_3plus_months}")

if customers_3plus_months >= 2000:
    print("DECISION: PROCEED with Olist.")
elif customers_3plus_months >= 500:
    print("DECISION: MARGINAL. Consider quarterly windows.")
else:
    print("DECISION: FALLBACK to Online Retail II.")


Distinct active months per customer:
purchase_month
1     94299
2      1679
3        96
4        17
5         1
6         3
10        1
Name: count, dtype: int64

Customers with >=3 active months: 118
DECISION: FALLBACK to Online Retail II.


In [22]:
# CELL 8: Load Online Retail II
# Add the dataset via Kaggle's "Add Input" - search "Online Retail II"
# Then list CSVs:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        if 'online' in filename.lower() or 'retail' in filename.lower():
            print(os.path.join(dirname, filename))

/kaggle/input/datasets/mashlyn/online-retail-ii-uci/online_retail_II.csv


In [23]:
# CELL 9: Load Online Retail II
df = pd.read_csv('/kaggle/input/datasets/mashlyn/online-retail-ii-uci/online_retail_II.csv', 
                 encoding='latin-1')
print(f"Shape: {df.shape}")
print(df.head())
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nDate range: {df['InvoiceDate'].min()} to {df['InvoiceDate'].max()}")

Shape: (1067371, 8)
  Invoice StockCode                          Description  Quantity  \
0  489434     85048  15CM CHRISTMAS GLASS BALL 20 LIGHTS        12   
1  489434    79323P                   PINK CHERRY LIGHTS        12   
2  489434    79323W                  WHITE CHERRY LIGHTS        12   
3  489434     22041         RECORD FRAME 7" SINGLE SIZE         48   
4  489434     21232       STRAWBERRY CERAMIC TRINKET BOX        24   

           InvoiceDate  Price  Customer ID         Country  
0  2009-12-01 07:45:00   6.95      13085.0  United Kingdom  
1  2009-12-01 07:45:00   6.75      13085.0  United Kingdom  
2  2009-12-01 07:45:00   6.75      13085.0  United Kingdom  
3  2009-12-01 07:45:00   2.10      13085.0  United Kingdom  
4  2009-12-01 07:45:00   1.25      13085.0  United Kingdom  

Columns: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']

Date range: 2009-12-01 07:45:00 to 2011-12-09 12:50:00


In [24]:
# CELL 10: Basic cleaning
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df = df.dropna(subset=['Customer ID'])
df['Customer ID'] = df['Customer ID'].astype(int).astype(str)
df['TotalPrice'] = df['Quantity'] * df['Price']

# Remove cancelled/return transactions
df = df[~df['Invoice'].astype(str).str.startswith('C')]

print(f"Cleaned shape: {df.shape}")
print(f"Unique customers: {df['Customer ID'].nunique()}")
print(f"Unique invoices: {df['Invoice'].nunique()}")

Cleaned shape: (805620, 9)
Unique customers: 5881
Unique invoices: 36975


In [25]:
# CELL 11: VIABILITY CHECK - Online Retail II
df['purchase_month'] = df['InvoiceDate'].dt.to_period('M')
active_months = df.groupby('Customer ID')['purchase_month'].nunique()
print("Distinct active months per customer:")
print(active_months.value_counts().sort_index().head(20))

customers_3plus = (active_months >= 3).sum()
print(f"\nCustomers with >=3 active months: {customers_3plus}")
print(f"Out of {df['Customer ID'].nunique()} total customers")
print(f"Percentage: {customers_3plus/df['Customer ID'].nunique()*100:.1f}%")

Distinct active months per customer:
purchase_month
1     1781
2     1070
3      661
4      494
5      387
6      281
7      198
8      182
9      150
10     115
11      92
12      75
13      52
14      58
15      41
16      47
17      34
18      28
19      33
20      27
Name: count, dtype: int64

Customers with >=3 active months: 3030
Out of 5881 total customers
Percentage: 51.5%


In [26]:
# CELL 12: Filter to trajectory-eligible customers only
eligible_customers = active_months[active_months >= 3].index
df_traj = df[df['Customer ID'].isin(eligible_customers)].copy()
print(f"Filtered shape: {df_traj.shape}")
print(f"Eligible customers: {df_traj['Customer ID'].nunique()}")

Filtered shape: (709997, 10)
Eligible customers: 3030


In [27]:
# CELL 13: Define time boundaries for temporal split
df_traj['purchase_month'] = df_traj['InvoiceDate'].dt.to_period('M')
all_months = sorted(df_traj['purchase_month'].unique())
print(f"Total months in data: {len(all_months)}")
print(f"First month: {all_months[0]}")
print(f"Last month: {all_months[-1]}")

# Holdout: last 3 months
holdout_months = all_months[-3:]
train_months = all_months[:-3]
print(f"Training months: {train_months[0]} to {train_months[-1]} ({len(train_months)} months)")
print(f"Holdout months: {holdout_months[0]} to {holdout_months[-1]} ({len(holdout_months)} months)")

Total months in data: 25
First month: 2009-12
Last month: 2011-12
Training months: 2009-12 to 2011-09 (22 months)
Holdout months: 2011-10 to 2011-12 (3 months)
